# Intel Lab Dataset — Quick Plots Demo

Load data and generate the three key visualizations.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os, urllib.request, gzip

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 4)
plt.rcParams['figure.dpi'] = 100

In [ ]:
# Download dataset if needed
data_path = 'data.txt'
if not os.path.exists(data_path):
    print('Downloading...')
    url = 'https://db.csail.mit.edu/labdata/data.txt.gz'
    urllib.request.urlretrieve(url, 'data.txt.gz')
    with gzip.open('data.txt.gz', 'rb') as f_in:
        with open(data_path, 'wb') as f_out:
            f_out.write(f_in.read())

# Load
columns = ['date', 'time', 'epoch', 'moteid', 'temperature', 'humidity', 'light', 'voltage']
df = pd.read_csv(data_path, sep=r'\s+', names=columns, na_values=[''])
df['timestamp'] = pd.to_datetime(df['date'] + ' ' + df['time'], errors='coerce')
df = df.dropna(subset=['timestamp'])
df = df[df['moteid'].between(1, 54)]

print(f'{len(df):,} rows loaded')

## Plot 1: Readings per Sensor

In [ ]:
readings_per_mote = df.groupby('moteid').size().sort_values(ascending=False)

fig, ax = plt.subplots()
readings_per_mote.hist(bins=30, ax=ax)
ax.set_xlabel('Number of readings')
ax.set_ylabel('Number of sensors')
ax.set_title('Distribution of readings per sensor')
plt.tight_layout()
plt.savefig('audit_readings_per_sensor.png', dpi=150, bbox_inches='tight')
plt.show()

## Plot 2: Temperature Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

df['temperature'].hist(bins=100, ax=axes[0])
axes[0].set_xlabel('Temperature (°C)')
axes[0].set_ylabel('Count')
axes[0].set_title('Full range')
axes[0].axvline(x=40, color='red', linestyle='--', label='40°C')
axes[0].legend()

df['temperature'].hist(bins=100, ax=axes[1], range=(15, 35))
axes[1].set_xlabel('Temperature (°C)')
axes[1].set_ylabel('Count')
axes[1].set_title('Plausible range (15-35°C)')

plt.tight_layout()
plt.savefig('audit_temperature_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## Plot 3: Time Series (Selected Sensors)

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

for i, mote_id in enumerate([1, 49, 5]):
    mote_data = df[df['moteid'] == mote_id].sort_values('timestamp')
    axes[i].plot(mote_data['timestamp'], mote_data['temperature'], 'b-', alpha=0.5, markersize=1)
    axes[i].set_ylabel(f'Mote {mote_id}\nTemp (°C)')
    axes[i].set_ylim(15, 50)
    
    if mote_id == 49:
        last_valid = mote_data['timestamp'].max()
        axes[i].axvline(x=last_valid, color='red', linestyle='--', alpha=0.7)

axes[-1].set_xlabel('Timestamp')
axes[0].set_title('Temperature time series')

plt.tight_layout()
plt.savefig('audit_time_series.png', dpi=150, bbox_inches='tight')
plt.show()

## Plot 4: Correlation Matrix

In [ ]:
# Resample to 5-min intervals
df_5min = df.set_index('timestamp').groupby('moteid').resample('5min').agg({'temperature': 'mean'}).reset_index()
temp_pivot = df_5min.pivot(index='timestamp', columns='moteid', values='temperature')
valid_sensors = temp_pivot.count()[temp_pivot.count() > 1000].index
temp_pivot = temp_pivot[valid_sensors]

# Correlation for first 10 sensors
corr_matrix = temp_pivot.iloc[:, :10].corr()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='RdYlGn', vmin=0.8, vmax=1.0, ax=ax)
ax.set_title('Temperature correlation (Motes 1-10)')
plt.tight_layout()
plt.savefig('compare_correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## Plot 5: Temperature Spread

In [ ]:
temp_spread = temp_pivot.max(axis=1) - temp_pivot.min(axis=1)

fig, ax = plt.subplots(figsize=(12, 4))
temp_spread.hist(bins=100, ax=ax)
ax.set_xlabel('Temperature spread (°C)')
ax.set_ylabel('Count')
ax.set_title('Distribution of temperature spread between sensors')
ax.axvline(x=temp_spread.median(), color='green', linestyle='--', label=f'Median: {temp_spread.median():.2f}°C')
ax.legend()
plt.tight_layout()
plt.savefig('compare_spread_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## Plot 6: Anomaly Detection

In [ ]:
# Mote 1 analysis
mote1 = df[df['moteid'] == 1].sort_values('timestamp').set_index('timestamp')

# Fixed threshold
mote1['anomaly_fixed'] = (mote1['temperature'] < 15) | (mote1['temperature'] > 35)

# Z-score
rolling_mean = mote1['temperature'].rolling('1h', center=True).mean()
rolling_std = mote1['temperature'].rolling('1h', center=True).std()
mote1['z_score'] = (mote1['temperature'] - rolling_mean) / rolling_std
mote1['anomaly_zscore'] = mote1['z_score'].abs() > 3

# Plot
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(mote1.index, mote1['temperature'], 'b-', alpha=0.3, label='Temperature', markersize=1)

fixed_anomalies = mote1[mote1['anomaly_fixed']]
ax.scatter(fixed_anomalies.index, fixed_anomalies['temperature'], 
           c='red', s=10, alpha=0.5, label=f'Fixed threshold ({len(fixed_anomalies):,})')

zscore_only = mote1[mote1['anomaly_zscore'] & ~mote1['anomaly_fixed']]
ax.scatter(zscore_only.index, zscore_only['temperature'],
           c='green', s=20, alpha=0.7, label=f'Z-score only ({len(zscore_only):,})', marker='^')

ax.axhline(y=15, color='orange', linestyle='--', alpha=0.5)
ax.axhline(y=35, color='orange', linestyle='--', alpha=0.5)
ax.set_xlabel('Timestamp')
ax.set_ylabel('Temperature (°C)')
ax.set_title('Mote 1: Anomaly Detection Comparison')
ax.legend(loc='upper right')
ax.set_ylim(10, 50)

plt.tight_layout()
plt.savefig('anomalies.png', dpi=150, bbox_inches='tight')
plt.show()